In [1]:
from chemplus.unicore import docking
from chemplus import mol_df
ucc_path = "C:/Users/USER/software/ucc-1.3.1/bin/ucc.bat"

# Set parameters for docking
The parameter **serial = True** allows docking calculations on a single node of the supercomputer, but parallelizes the computations of a single docking. Set this parameter if you need to perform docking quickly for a small number of compounds with high exhaustiveness.

The parameter **serial = False** allows docking calculations on more than one node using the MPI version. Set this parameter if you have a large number of compounds.

The parameter **sync = False** allows the docking process to run without waiting for its completion. In this case, you will need to 
manually check if the docking has finished.

The parameter **sync = True** makes the code wait for the docking to complete and automatically retrieves the results from the supercomputer after completion.

In [2]:
server_work_dir = "~/projects/test_docking"
server_executable = "/share/bio/data/miniconda3/envs/chem/bin/python"

local_receptor = "test/hr1(2).pdbqt"
local_pdb_receptor = "test/hr1(2).pdb"
local_sdf = "test/sm_compounds.sdf"
local_config = "test/config_file.txt"
local_out_dir = "test"

cpu_count = 80
sync = False
serial = False

#multiruns
docking_runs_num = 2
local_out_sdf_runs = "test/sm_compound_2runs.sdf"

# Run docking

In [3]:
df = mol_df.df_from_sdf(local_sdf)
print(f"Number of molecules: {df.shape[0]}")
if docking_runs_num > 1:
    df["Docking run"] = [list(map(str, range(1, docking_runs_num + 1)))]*df.shape[0]
    df = df.explode("Docking run", ignore_index=True)
    df["Name"] = df["Name"] + "_run-" + df["Docking run"]
    mol_df.df_to_sdf(df, local_out_sdf_runs)
    run_sdf = local_out_sdf_runs
else:
    run_sdf = local_sdf
job_id_file = docking.unicore_dock(ucc_path, "SKIF_GRID_CIS", server_executable, server_work_dir, cpu_count, 
                                   local_out_dir, local_receptor, local_pdb_receptor, local_config, run_sdf, sync=sync, serial=serial)
print(f"Job id file:\n{job_id_file}")

Number of molecules: 20
Job id file:
C:\Users\USER\OneDrive\Documents\work\chemnotes\Docking\test\b24fc52b-7589-4991-86a4-7e2b76156605.job


# Checking docking completion and retrieving results
(if sync = False)

In [7]:
#job_id_file = r"C:\Users\USER\OneDrive\Documents\work\SARS-CoV-2 Mpro\SH-4-54\.\caa04ece-b65c-4b19-a38e-7053a02b27c0.job"

docking.get_docking_results(ucc_path, job_id_file, local_out_dir)

'SUCCESSFUL exit code: 0'